# 02 · Analyze — stats + figures from `cases.csv`

**Inputs:** `data/cases.csv` (one row per case, **opinion-only** features).  
**What it does:** derives per-sentence framing rates, compares male- vs female-victim cases **within each stratum** (criminal / civil) using Mann-Whitney U, Cliff's δ with 500-resample bootstrap CIs, and Benjamini–Hochberg FDR (+ Bonferroni) across detectors.  
**Outputs:** `output/headline_stats.csv`, `output/forest_stats.csv`, and `figure1_corpus_composition.png`, `figure2_direct_violence_by_gender.png`, `hero_act_vs_actor.png`, `09_forest_plot.png`.

This notebook runs end-to-end from the committed `cases.csv` — no raw opinions or network access needed.

In [ ]:
# --- portable paths + imports (no hardcoded absolute paths) ---
from pathlib import Path
import pandas as pd, numpy as np, math
import matplotlib as mpl, matplotlib.pyplot as plt
from scipy import stats

REPO   = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
DATA   = REPO / 'data'
OUTPUT = REPO / 'output'
OUTPUT.mkdir(parents=True, exist_ok=True)
mpl.rcParams['font.family'] = 'DejaVu Sans'
print('REPO :', REPO)

In [ ]:
# --- functions defined up front ---
def cliffs_d(a, b):
    """Cliff's delta. Sign convention: positive => group a tends to exceed group b."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    if len(a) == 0 or len(b) == 0:
        return float('nan')
    return ((a[:, None] > b[None, :]).sum() - (a[:, None] < b[None, :]).sum()) / (len(a) * len(b))

def bootstrap_ci(a, b, n_boot=500, seed=42):
    """500-resample bootstrap 95% CI for Cliff's delta."""
    rng = np.random.default_rng(seed)
    boots = [cliffs_d(rng.choice(a, len(a), True), rng.choice(b, len(b), True)) for _ in range(n_boot)]
    return float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))

def stratum_stats(df, var, stratum=None):
    """Mann-Whitney U + Cliff's delta (male vs female) within an optional stratum."""
    sub = df if stratum is None else df[df['stratum'] == stratum]
    male   = sub[sub['gender'] == 'male'][var].dropna().values
    female = sub[sub['gender'] == 'female'][var].dropna().values
    if len(male) < 2 or len(female) < 2:
        return None
    U, p = stats.mannwhitneyu(male, female, alternative='two-sided')
    d = cliffs_d(male, female)
    lo, hi = bootstrap_ci(male, female)
    return dict(n_male=len(male), n_female=len(female),
                median_male=float(np.median(male)), median_female=float(np.median(female)),
                U=float(U), p=float(p), cliffs_d=float(d), ci_lo=lo, ci_hi=hi)

def fdr_bh(pvals):
    """Benjamini-Hochberg FDR-adjusted q-values."""
    p = np.asarray(pvals, float); n = len(p)
    order = np.argsort(p); ranked = p[order]
    q = ranked * n / np.arange(1, n + 1)
    for i in range(n - 2, -1, -1):
        q[i] = min(q[i], q[i + 1])
    q = np.minimum(q, 1.0)
    out = np.empty(n); out[order] = q
    return out.tolist()

# Palette: pink = female, blue = male (consistent across every figure)
PINK, PINK_F = '#E4889B', '#F2C4CE'
BLUE, BLUE_F = '#7E9FC0', '#C3D2E2'
NAVY, GREY   = '#2B3A55', '#5A6472'


In [ ]:
# --- load opinion-only data + derive rates + diagnostics ---
df = pd.read_csv(DATA / 'cases.csv')
df['gender']  = df['folder_victim_gender_label']
df['stratum'] = np.where(df['is_govt_prosecution'] == True, 'criminal', 'civil')
for c in [c for c in df.columns if c.startswith('sa_')]:
    df[f'{c}_rate'] = df[c] / df['n_sentences'].replace(0, np.nan)

print('cases:', len(df), '| columns:', df.shape[1])
print('gender x stratum:')
print(pd.crosstab(df['gender'], df['stratum']))

## Headline effect sizes

In [ ]:
# --- headline_stats.csv : 5 measures x 3 strata ---
VAR = {'direct_violence':'sa_direct_violence_term_rate', 'perp_naming':'sa_perp_ref_name_rate',
       'crime_vocab':'empath_crime', 'victim_blame':'sa_victim_blame_direct_rate',
       'affect':'vader_para_mean_compound'}
rows = []
for label, var in VAR.items():
    for strat in ['full', 'criminal', 'civil']:
        s = stratum_stats(df, var, None if strat == 'full' else strat)
        if s: rows.append(dict(variable=label, var_col=var, stratum=strat, **s))
hl = pd.DataFrame(rows)[['variable','var_col','stratum','n_male','n_female',
                         'median_male','median_female','U','p','cliffs_d','ci_lo','ci_hi']]
hl.to_csv(OUTPUT / 'headline_stats.csv', index=False)
print(hl[hl.variable.isin(['direct_violence','perp_naming'])].to_string(index=False))

## Forest table across all framing detectors

In [ ]:
# --- forest_stats.csv : 15 framing detectors, criminal stratum, with FDR + Bonferroni ---
FOREST = [('sa_direct_violence_term_rate','Direct violence terms'),('sa_perp_ref_name_rate','Perpetrator naming'),
('empath_crime','Empath: crime vocabulary'),('sa_victim_blame_direct_rate','Victim-blaming language'),
('vader_para_mean_compound','VADER affect (paragraph)'),('sa_perp_active_agency_rate','Perpetrator as active agent'),
('sa_victim_passive_object_rate','Victim as passive object'),('empath_violence','Empath: violence vocabulary'),
('sa_clinical_legal_term_rate','Clinical/sterilizing terms'),('sa_credibility_challenge_rate','Credibility challenges'),
('sa_hedging_allegation_rate','Hedging density'),('sa_minimizing_term_rate','Minimizing language'),
('sa_procedural_deflection_rate','Procedural deflection'),('sa_perp_honorific_rate','Perpetrator honorifics'),
('pct_passive_aux','Passive voice density')]
forest = []
for var, label in FOREST:
    s = stratum_stats(df, var, 'criminal'); s['label'] = label; s['var'] = var; forest.append(s)
q_bh = fdr_bh([s['p'] for s in forest]); n_tests = len(forest)
fr = []
for s, q in zip(forest, q_bh):
    pb = min(1.0, s['p'] * n_tests)
    fr.append(dict(label=s['label'], var=s['var'], n_male=s['n_male'], n_female=s['n_female'],
        U=s['U'], p=s['p'], cliffs_d=s['cliffs_d'], ci_lo=s['ci_lo'], ci_hi=s['ci_hi'],
        q_fdr_bh=q, p_bonf=pb, fdr_sig=bool(q < 0.05), bonf_sig=bool(pb < 0.05)))
fs = pd.DataFrame(fr)
fs.to_csv(OUTPUT / 'forest_stats.csv', index=False)
print('FDR-significant detectors:', fs[fs.fdr_sig].label.tolist())

## Figures

In [ ]:
# --- Figure 1 : corpus composition (structural; unaffected by text scope) ---
ct = pd.crosstab(df['stratum'].str.capitalize(), df['gender'].str.capitalize())[['Female','Male']]
fig, ax = plt.subplots(figsize=(7.6, 4.8)); fig.patch.set_facecolor('white')
x = np.arange(len(ct.index)); w = 0.36
bf = ax.bar(x - w/2, ct['Female'], w, color=PINK_F, edgecolor=PINK, linewidth=1.2, label='Female victim')
bm = ax.bar(x + w/2, ct['Male'],   w, color=BLUE_F, edgecolor=BLUE, linewidth=1.2, label='Male victim')
for bars in (bf, bm):
    for b in bars:
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.5, f'{int(b.get_height())}',
                ha='center', va='bottom', fontsize=10, color=GREY)
ax.set_xticks(x); ax.set_xticklabels(ct.index, color=GREY)
ax.set_ylabel('Number of cases', color=GREY); ax.set_xlabel('Case type', color=GREY)
ax.set_title('Figure 1. Corpus composition by victim gender and case type',
             loc='left', fontsize=13, fontweight='bold', color=NAVY, pad=24)
ax.text(0, 1.04, 'n = 97 U.S. judicial opinions; female victims concentrate in criminal prosecutions, male victims in civil suits.',
        transform=ax.transAxes, fontsize=9.5, color=GREY, style='italic')
ax.legend(loc='upper right', frameon=False, labelcolor=GREY)
ax.spines[['top','right']].set_visible(False); ax.spines[['left','bottom']].set_color(GREY)
ax.tick_params(colors=GREY); ax.yaxis.grid(True, color='#E6E8EC', linewidth=0.8); ax.set_axisbelow(True)
ax.set_ylim(0, max(ct.values.max()+6, 40))
plt.tight_layout(); plt.savefig(OUTPUT / 'figure1_corpus_composition.png', dpi=200, bbox_inches='tight', facecolor='white'); plt.close()
print('saved figure1_corpus_composition.png')

In [ ]:
# --- Figure 2 : direct-violence rate, criminal stratum ---
crim = df[df['stratum'] == 'criminal']
f = crim[crim['gender']=='female']['sa_direct_violence_term_rate'].values
m = crim[crim['gender']=='male']['sa_direct_violence_term_rate'].values
d_dv = cliffs_d(m, f)
fig, ax = plt.subplots(figsize=(9, 5.6)); fig.patch.set_facecolor('white')
bp = ax.boxplot([f, m], positions=[0,1], widths=0.5, patch_artist=True, showfliers=False,
   medianprops=dict(color='white', linewidth=2.5), whiskerprops=dict(color=GREY),
   capprops=dict(color=GREY), boxprops=dict(linewidth=1.2))
for patch,c,e in zip(bp['boxes'], [PINK_F,BLUE_F], [PINK,BLUE]): patch.set_facecolor(c); patch.set_edgecolor(e)
rng = np.random.default_rng(7)
for i,(d,e) in enumerate(zip([f,m], [PINK,BLUE])):
    ax.scatter(np.full(len(d),i)+rng.uniform(-0.08,0.08,len(d)), d, s=30, color=e, alpha=0.65,
               edgecolor='white', linewidth=0.5, zorder=3)
ax.set_xticks([0,1]); ax.set_xticklabels([f'Female victim\nn = {len(f)}', f'Male victim\nn = {len(m)}'], fontsize=12, color=GREY)
ax.set_ylabel('Direct-violence-term rate\n(share of sentences)', fontsize=12, color=GREY)
ax.set_title('Figure 2. Direct-violence vocabulary by victim gender, criminal cases',
             loc='left', fontsize=14.5, color=NAVY, fontweight='bold', pad=26)
ax.text(0, 1.03, "Each point is one judicial opinion. Cliff's $\\delta$ = %.2f, $q_{FDR}$ < 0.01 — female-victim opinions name the violence directly more often." % d_dv,
        transform=ax.transAxes, fontsize=10, color=GREY, style='italic')
ax.spines[['top','right']].set_visible(False); ax.spines[['left','bottom']].set_color(GREY)
ax.tick_params(colors=GREY); ax.yaxis.grid(True, color='#E6E8EC', linewidth=0.8); ax.set_axisbelow(True); ax.set_ylim(bottom=-0.02)
plt.tight_layout(); plt.savefig(OUTPUT / 'figure2_direct_violence_by_gender.png', dpi=200, bbox_inches='tight', facecolor='white'); plt.close()
print(f'saved figure2  (delta = {d_dv:+.2f})')

In [ ]:
# --- Hero : act vs. actor (the headline finding) ---
f_pn = crim[crim['gender']=='female']['sa_perp_ref_name_rate'].values
m_pn = crim[crim['gender']=='male']['sa_perp_ref_name_rate'].values
d_pn = cliffs_d(m_pn, f_pn)
fig, axes = plt.subplots(1, 2, figsize=(12, 6.4)); fig.patch.set_facecolor('white')
def panel(ax, fdat, mdat, title, ylab, note):
    bp = ax.boxplot([fdat, mdat], positions=[0,1], widths=0.5, patch_artist=True, showfliers=False,
        medianprops=dict(color='white', linewidth=2.5), whiskerprops=dict(color=GREY),
        capprops=dict(color=GREY), boxprops=dict(linewidth=1.2))
    for patch,c,e in zip(bp['boxes'], [PINK_F,BLUE_F], [PINK,BLUE]): patch.set_facecolor(c); patch.set_edgecolor(e)
    rng = np.random.default_rng(7)
    for i,(d,e) in enumerate(zip([fdat,mdat], [PINK,BLUE])):
        ax.scatter(np.full(len(d),i)+rng.uniform(-0.06,0.06,len(d)), d, s=26, color=e, alpha=0.65, edgecolor='white', linewidth=0.5, zorder=3)
    ax.set_xticks([0,1]); ax.set_xticklabels([f'Female\nn={len(fdat)}', f'Male\nn={len(mdat)}'], fontsize=12, color=GREY)
    ax.set_ylabel(ylab, fontsize=12, color=GREY)
    ax.set_title(title, fontsize=14, color=NAVY, fontweight='bold', pad=24, loc='left')
    ax.text(0, 1.02, note, transform=ax.transAxes, fontsize=10.5, color=GREY, style='italic')
    ax.spines[['top','right']].set_visible(False); ax.spines[['left','bottom']].set_color(GREY)
    ax.tick_params(colors=GREY); ax.set_ylim(bottom=-0.02); ax.yaxis.grid(True, color='#E6E8EC', linewidth=0.8); ax.set_axisbelow(True)
panel(axes[0], f, m, 'Naming the act', 'Direct-violence-term rate\n(share of sentences)',
      r'$\delta=%.2f$, $q_{FDR}<0.01$  ·  female victims higher' % d_dv)
panel(axes[1], f_pn, m_pn, 'Naming the actor', 'Perpetrator-naming rate\n(share of sentences)',
      r'$\delta=%+.2f$, $q_{FDR}<0.01$  ·  male victims higher' % d_pn)
fig.suptitle('In criminal sexual-assault opinions, language centers the act for women and the actor for men',
             fontsize=15.5, fontweight='bold', color=NAVY, x=0.012, ha='left', y=1.0)
fig.text(0.012, 0.945, 'Criminal stratum only (n = 50 opinions). Each point is one judicial opinion.',
         fontsize=11, color=GREY, style='italic', ha='left')
plt.tight_layout(rect=[0,0,1,0.88]); plt.savefig(OUTPUT / 'hero_act_vs_actor.png', dpi=240, bbox_inches='tight', facecolor='white'); plt.close()
print(f'saved hero  (act delta={d_dv:+.2f}, actor delta={d_pn:+.2f})')

In [ ]:
# --- Figure 3 : forest plot of all detectors (criminal stratum) ---
fsp = fs.sort_values('cliffs_d')
fig, ax = plt.subplots(figsize=(10, 7.2)); fig.patch.set_facecolor('white')
ax.axvspan(-0.147, 0.147, color='#EEF0F3', zorder=0); ax.axvline(0, color=GREY, linewidth=1, zorder=1)
for y,(_,r) in enumerate(fsp.iterrows()):
    sig = r['fdr_sig']; col = NAVY if sig else '#9AA3B2'
    ax.plot([r['ci_lo'], r['ci_hi']], [y,y], color=col, linewidth=2 if sig else 1.4, zorder=2)
    ax.scatter([r['cliffs_d']], [y], s=90 if sig else 55, color=col, zorder=3, edgecolor='white', linewidth=1)
    if sig: ax.text(r['ci_hi']+0.03, y, f"q={r['q_fdr_bh']:.3f} *", va='center', fontsize=9, color=NAVY, fontweight='bold')
ax.set_yticks(range(len(fsp))); ax.set_yticklabels(fsp['label'], fontsize=10.5, color=GREY)
ax.set_xlabel("Cliff's δ  (sign = male − female;  negative = female-victim opinions higher)", fontsize=11, color=GREY)
ax.set_xlim(-1, 1)
ax.set_title('Figure 3. Gender framing gaps across detectors, criminal cases', fontsize=14.5, color=NAVY, fontweight='bold', loc='left', pad=24)
ax.text(0, 1.02, "Markers = Cliff's δ; bars = 500-resample bootstrap 95% CIs. Grey band = negligible (±0.147). * = significant after Benjamini–Hochberg FDR.",
        transform=ax.transAxes, fontsize=9.5, color=GREY, style='italic')
ax.spines[['top','right']].set_visible(False); ax.spines[['left','bottom']].set_color(GREY); ax.tick_params(colors=GREY)
plt.tight_layout(); plt.savefig(OUTPUT / '09_forest_plot.png', dpi=200, bbox_inches='tight', facecolor='white'); plt.close()
print('saved 09_forest_plot.png')

Done — all stats tables and figures written to `output/`.